# GRU — Traffic Congestion Prediction (Single Model, All 4 Roads)
One GRU trained on all roads. Road identity is encoded as a feature so the model predicts congestion independently for each road.

In [23]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

In [24]:
!nvidia-smi

Tue Apr 28 18:12:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |     421MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:
from google.colab import drive
drive.mount('/content/drive')

# then upload the data folder to colab /content folder, or the working directory for temporal stoage during that session

## 1. Load & Encode

In [26]:
df = pd.read_csv('/content/data/feature_store/traffic_for_lecture_system.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values(['road', 'datetime']).reset_index(drop=True)

ROADS = sorted(df['road'].unique())

road_le  = LabelEncoder()
label_le = LabelEncoder()
df['road_enc'] = road_le.fit_transform(df['road'])
df['label']    = label_le.fit_transform(df['congestion'])

print('Roads  :', ROADS)
print('Classes:', label_le.classes_)

Roads  : ['Bombo Road', 'Northern Bypass', 'Sir Apolo Road', 'University Road']
Classes: ['Low' 'Normal' 'Very high']


## 2. Sliding-Window Sequences — split per road so all roads appear in test

In [27]:
SEQ_LEN   = 16
FEAT_COLS = ['road_enc', 'month', 'hour', 'day_of_week',
             'is_weekend', 'is_rush_hour', 'estimated_vehicle_count']

scaler = StandardScaler()
df[FEAT_COLS] = scaler.fit_transform(df[FEAT_COLS])

train_X, train_y = [], []
test_X,  test_y  = [], []
test_road_labels = []          # road name per test sample

for road in ROADS:
    rdf    = df[df['road'] == road].reset_index(drop=True)
    feats  = rdf[FEAT_COLS].values
    labels = rdf['label'].values

    X, y = [], []
    for i in range(SEQ_LEN, len(feats)):
        X.append(feats[i-SEQ_LEN:i])
        y.append(labels[i])

    split = int(0.8 * len(y))
    train_X.extend(X[:split]);  train_y.extend(y[:split])
    test_X.extend(X[split:]);   test_y.extend(y[split:])
    test_road_labels.extend([road] * (len(y) - split))
    print(f'{road}: train={split}  test={len(y)-split}')

train_X = np.array(train_X, dtype=np.float32)
train_y = np.array(train_y, dtype=np.int64)
test_X  = np.array(test_X,  dtype=np.float32)
test_y  = np.array(test_y,  dtype=np.int64)
test_road_labels = np.array(test_road_labels)

Bombo Road: train=8051  test=2013
Northern Bypass: train=8051  test=2013
Sir Apolo Road: train=8051  test=2013
University Road: train=8051  test=2013


In [28]:
class TrafficDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X)
        self.y = torch.tensor(y)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(TrafficDataset(train_X, train_y), batch_size=128, shuffle=True)
test_loader  = DataLoader(TrafficDataset(test_X,  test_y),  batch_size=128, shuffle=False)

## 3. GRU Model

In [29]:
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes, num_layers=2):
        super().__init__()
        self.gru  = nn.GRU(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=0.3)
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        _, h = self.gru(x)
        return self.head(h[-1])

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = GRUClassifier(input_size=len(FEAT_COLS), hidden_size=64,
                           num_classes=len(label_le.classes_)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
print(model)
print('Device:', DEVICE)

GRUClassifier(
  (gru): GRU(7, 64, num_layers=2, batch_first=True, dropout=0.3)
  (head): Linear(in_features=64, out_features=3, bias=True)
)
Device: cuda


## 4. Train

In [30]:
EPOCHS = 30

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        loss = F.cross_entropy(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 5 == 0:
        print(f'Epoch {epoch:3d} | Loss: {total_loss/len(train_loader):.4f}')

Epoch   5 | Loss: 0.1598
Epoch  10 | Loss: 0.1355
Epoch  15 | Loss: 0.1188
Epoch  20 | Loss: 0.1112
Epoch  25 | Loss: 0.1060
Epoch  30 | Loss: 0.0985


## 5. Evaluate — overall + per-road

In [31]:
model.eval()
all_preds = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        preds = model(X_batch.to(DEVICE)).argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)

all_preds = np.array(all_preds)

print('=== Overall ===')
print(classification_report(test_y, all_preds, target_names=label_le.classes_, zero_division=0))

for road in ROADS:
    mask = test_road_labels == road
    print(f'\n=== {road} ===')
    print(classification_report(test_y[mask], all_preds[mask],
                                target_names=label_le.classes_, zero_division=0))

=== Overall ===
              precision    recall  f1-score   support

         Low       0.97      0.99      0.98      1803
      Normal       0.97      0.96      0.97      4095
   Very high       0.95      0.96      0.95      2154

    accuracy                           0.97      8052
   macro avg       0.96      0.97      0.97      8052
weighted avg       0.97      0.97      0.97      8052


=== Bombo Road ===
              precision    recall  f1-score   support

         Low       0.96      0.99      0.97       438
      Normal       0.94      0.93      0.94      1014
   Very high       0.91      0.90      0.91       561

    accuracy                           0.94      2013
   macro avg       0.94      0.94      0.94      2013
weighted avg       0.94      0.94      0.94      2013


=== Northern Bypass ===
              precision    recall  f1-score   support

         Low       0.97      0.99      0.98       718
      Normal       0.99      0.98      0.98      1241
   Very high  

## 6. Save

In [36]:
import h5py
import os

path = '/content/drive/MyDrive/c/ai/traffic/uncompressed/'
os.makedirs(path, exist_ok=True)

file_path = os.path.join(path, 'gru_traffic_model.h5')

with h5py.File(file_path, 'w') as f:
    for name, param in model.state_dict().items():
        f.create_dataset(name, data=param.cpu().numpy())

print(' ◱ ✓ Saved:', file_path)


Saved: /content/drive/MyDrive/c/ai/traffic/uncompressed/gru_traffic_model.h5


In [35]:
import os

# Create the directory if it doesn't exist
save_dir = '/content/drive/MyDrive/c/ai/traffic/uncompressed/'
os.makedirs(save_dir, exist_ok=True)

# Save your data arrays for compression
np.savez(os.path.join(save_dir, 'training_data.npz'),
         train_X=train_X, 
         train_y=train_y,
         test_X=test_X, 
         test_y=test_y,
         test_road_labels=test_road_labels)
print(f' ✓ Training data saved to: {save_dir}/training_data.npz')

# Also save the label encoder classes (optional but useful)
import pickle
with open(os.path.join(save_dir, 'label_encoder.pkl'), 'wb') as f:
    pickle.dump(label_le, f)
print(f' ✓ Label encoder saved to: {save_dir}/label_encoder.pkl')

✅ Training data saved to: /content/drive/MyDrive/c/ai/traffic/uncompressed//training_data.npz
✅ Label encoder saved to: /content/drive/MyDrive/c/ai/traffic/uncompressed//label_encoder.pkl
